In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor
from PIL import Image
from activation_store import ProbingDataset
from torch.utils.data import DataLoader
import torch
import torch.multiprocessing as mp
from tqdm import tqdm
import threading
from queue import Queue

import os
import matplotlib.pyplot as plt
import numpy as np
import json

# Load SAEs and hooked model for all 12 layers
from dictionary_learning.utils import load_dictionary
from hooked_model import HookedModel

In [ ]:
device = 'cuda:6'  # Use one GPU for SAE inference

# Load SAEs for CLS tokens (existing code)
saes_cls = {}
configs_cls = {}
for layer in range(12):
    dir = f"/Checkpoints/SAE/Layer-wise/CLS/BatchTopK-orig_top-128/ef_8/layer_{layer}/trainer_0"
    sae, config = load_dictionary(dir, device)
    saes_cls[layer] = sae
    configs_cls[layer] = config
    print(f"CLS Layer {layer}: k={config['trainer']['k']}, dict_size={config['trainer']['dict_size']}")

# Load SAEs for Image tokens (you'll need to update the path)
saes_img = {}
configs_img = {}
for layer in range(12):
    dir = f"/Checkpoints/SAE/Layer-wise/Image/BatchTopK-orig_top-128/ef_8/layer_{layer}/trainer_0"  # Update this path
    sae, config = load_dictionary(dir, device)
    saes_img[layer] = sae
    configs_img[layer] = config
    print(f"Image Layer {layer}: k={config['trainer']['k']}, dict_size={config['trainer']['dict_size']}")

# Load hooked model with all layer hook points
hook_points = [f"layer-{i}_resid-post" for i in range(12)]
hooked_model = HookedModel(
    "openai/clip-vit-base-patch32", 
    hook_points=hook_points, 
    device=device
)

In [ ]:
# Load one image from your local dataset folder

image_folder = "/datasets/SAE-Probing_filtered_85"

image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']





# GPT-5 testbed
image_folder = "/datasets/SAE-Probing_filtered_85"
jsonl_path = "/SAE/circuit_tracing/concept_set/extracted_concepts_gpt5.jsonl"
image_files = []
with open(jsonl_path, 'r') as f:
    for line in f:
        data = json.loads(line.strip())
        # Get basename for matching with files in folder
        image_path = data['image_id']
        image_files.append(os.path.basename(image_path))
print(f"Loaded {len(image_files)} image filenames from .jsonl")
# GPT-5 testbed


if not image_files:
    print(f"No images found in {image_folder}")
else:
    print(f"Found {len(image_files)} images in the folder")
    
    # Load the first image (or you can specify which one)
    # image_index = 2501
    # image_index = 2505
    # image_index = 1679
    image_index = 2515
    # image_index = 3027
    image_path = os.path.join(image_folder, image_files[image_index])
    image = Image.open(image_path).convert('RGB')

    print(f"Loaded image: {image_files[image_index]}")
    print(f"Image size: {image.size}")
    
    # Display the image
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"Sample image: {image_files[image_index]}")
    plt.axis('off')
    plt.show()

In [ ]:
# # Get activations using the hooked model's method
# # Pass the raw PIL image, not the preprocessed tensor
# embeddings = hooked_model.get_activations_from_image(image)

# print(f"Got activations for {len(embeddings)} layers")
# for layer_name, act in embeddings.items():
#     print(f"{layer_name}: {act.shape}")


# Get activations using the hooked model's method
# Pass the raw PIL image, not the preprocessed tensor
embeddings = hooked_model.get_activations_from_image(image)

# # Normalize embeddings to match training setup
# normalized_embeddings = {}
# for layer_name, act in embeddings.items():
#     # Apply L2 normalization along the last dimension (feature dimension)
#     normalized_embeddings[layer_name] = act / act.norm(dim=-1, keepdim=True)

# # Replace original embeddings with normalized ones
# embeddings = normalized_embeddings

print(f"Got normalized activations for {len(embeddings)} layers")
for layer_name, act in embeddings.items():
    print(f"{layer_name}: {act.shape}")

In [ ]:
# Load frequent feature indices for each layer
freq_idx_cls = {}
freq_idx_img = {}

for layer in range(12):
    cls_path = f"/Checkpoints/SAE/Layer-wise/CLS/BatchTopK-orig_top-128/ef_8/layer_{layer}/trainer_0/freq_feat_idx_.pt"
    img_path = f"/Checkpoints/SAE/Layer-wise/Image/BatchTopK-orig_top-128/ef_8/layer_{layer}/trainer_0/freq_feat_idx_.pt"
    freq_idx_cls[layer] = torch.load(cls_path).cpu().numpy()
    freq_idx_img[layer] = torch.load(img_path).cpu().numpy()

In [ ]:
# Load concept files for each layer
concept_files_path_cls = "/dissect/CLS/BatchTopK-orig_top-128/ef-8/top1_concepts_gpt5-all"
concept_files_path_img = "/dissect/Image/BatchTopK-orig_top-128/ef-8/top1_concepts_meanpooled_gpt5-all"

layer_concepts_cls = {}
layer_concepts_img = {}

print("Loading concept files for CLS and Image tokens...")
for layer in range(12):
    # CLS concepts
    concept_file_cls = f"{concept_files_path_cls}/layer_{layer}_top1_concepts.txt"
    try:
        with open(concept_file_cls, 'r') as f:
            concepts_cls = [line.strip() for line in f.readlines()]
        layer_concepts_cls[layer] = concepts_cls
        print(f"CLS Layer {layer}: Loaded {len(concepts_cls)} concepts")
    except FileNotFoundError:
        print(f"Warning: CLS concept file not found for layer {layer}")
        layer_concepts_cls[layer] = []

    # Image concepts
    concept_file_img = f"{concept_files_path_img}/layer_{layer}_top1_concepts.txt"
    try:
        with open(concept_file_img, 'r') as f:
            concepts_img = [line.strip() for line in f.readlines()]
        layer_concepts_img[layer] = concepts_img
        print(f"Image Layer {layer}: Loaded {len(concepts_img)} concepts")
    except FileNotFoundError:
        print(f"Warning: Image concept file not found for layer {layer}")
        layer_concepts_img[layer] = []

print("Done loading concept files for CLS and Image tokens!")

In [ ]:
# Extract top activated features for CLS and Image tokens separately
top_k = 10  # Number of top features to extract per layer
layer_top_features_cls = {}
layer_top_features_img = {}

print("Extracting top activated features for CLS and Image tokens per layer...")
print("=" * 50)

for layer in range(12):
    layer_name = f"layer-{layer}_resid-post"
    layer_activations = embeddings[layer_name]

    # CLS token
    cls_activation = layer_activations[0, 0, :]
    sae_cls = saes_cls[layer]
    with torch.no_grad():
        feature_acts_cls = sae_cls.encode(cls_activation.unsqueeze(0)).squeeze(0)
        # Zero out frequent features
        feature_acts_cls[freq_idx_cls[layer]] = 0
        top_values_cls, top_indices_cls = torch.topk(feature_acts_cls, top_k)
        # Filter out zero activations
        nonzero_mask_cls = top_values_cls != 0
        layer_top_features_cls[layer] = {
            'indices': top_indices_cls[nonzero_mask_cls].cpu().numpy(),
            'values': top_values_cls[nonzero_mask_cls].cpu().numpy(),
            'feature_acts': feature_acts_cls.cpu().numpy()
        }

    # IMG tokens (mean pooled)
    img_activations = layer_activations[0, 1:, :]
    avg_img_activation = img_activations.mean(dim=0)
    sae_img = saes_img[layer]
    with torch.no_grad():
        # Encode each image token individually
        feature_acts_all_img = sae_img.encode(img_activations)  # [num_img_tokens, dict_size]
        # Average the feature activations
        feature_acts_img = feature_acts_all_img.mean(dim=0)  # [dict_size]
        # Zero out frequent features
        feature_acts_img[freq_idx_img[layer]] = 0
        top_values_img, top_indices_img = torch.topk(feature_acts_img, top_k)
        # Filter out zero activations
        nonzero_mask_img = top_values_img != 0
        layer_top_features_img[layer] = {
            'indices': top_indices_img[nonzero_mask_img].cpu().numpy(),
            'values': top_values_img[nonzero_mask_img].cpu().numpy(),
            'feature_acts': feature_acts_img.cpu().numpy()
        }

    print(f"Layer {layer}:")
    print(f"  CLS Top {top_k} indices: {layer_top_features_cls[layer]['indices']}")
    print(f"  CLS Top {top_k} values: {layer_top_features_cls[layer]['values']}")
    print(f"  IMG Top {top_k} indices: {layer_top_features_img[layer]['indices']}")
    print(f"  IMG Top {top_k} values: {layer_top_features_img[layer]['values']}")
    print()

In [ ]:
# Print top activated concepts for each layer (union of CLS and IMG tokens)
print("\nTop Activated Concepts for Each Layer (Union of CLS and IMG tokens):")
print("=" * 60)

for layer in range(12):
    print(f"\nLayer {layer}:")
    print("-" * 30)

    # Get top features and concepts for CLS and IMG tokens
    top_features_cls = layer_top_features_cls.get(layer, {})
    top_features_img = layer_top_features_img.get(layer, {})
    concepts_cls = layer_concepts_cls.get(layer, [])
    concepts_img = layer_concepts_img.get(layer, [])

    # Collect concepts from top features
    concept_set = set()
    # CLS
    if 'indices' in top_features_cls and concepts_cls:
        for feature_idx in top_features_cls['indices']:
            if feature_idx < len(concepts_cls):
                concept_set.add(concepts_cls[feature_idx])
    # IMG
    if 'indices' in top_features_img and concepts_img:
        for feature_idx in top_features_img['indices']:
            if feature_idx < len(concepts_img):
                concept_set.add(concepts_img[feature_idx])

    if concept_set:
        for i, concept in enumerate(sorted(concept_set), 1):
            print(f"  {i:2d}. {concept}")
    else:
        print("  No concepts available for this layer")

In [ ]:
## GPT-5 testbed
def extract_image_id(filename):
    """Extract image ID from filename like 'img_000005.jpeg' or full path."""
    # If full path, get basename
    base = os.path.basename(filename)
    # Remove extension
    name, _ = os.path.splitext(base)
    return name  # e.g., 'img_000005'

def load_ground_truth_concepts(file_path, image_filename):
    """Load ground truth concepts for a specific image (new dataset format)"""
    image_id = extract_image_id(image_filename)
    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line.strip())
            # Try to match by image_id (without extension)
            gt_image_id = extract_image_id(data.get('image_id', ''))
            if image_id == gt_image_id:
                return data.get('concepts', [])
    return None
## GPT-5 testbed

In [ ]:
import re

gt_concepts_file = "/concept_set/extracted_concepts_gpt5.jsonl"

# Get current image filename
current_image_filename = image_files[image_index]  # Use your selected image index

# Load ground truth concepts for this image
gt_concepts = load_ground_truth_concepts(gt_concepts_file, current_image_filename)

# Collect predicted concepts from top features (union of CLS and IMG)
predicted_concepts = set()
for layer in range(12):
    # CLS
    for feature_idx in layer_top_features_cls[layer]['indices']:
        if feature_idx < len(layer_concepts_cls[layer]):
            predicted_concepts.add(layer_concepts_cls[layer][feature_idx])
    # IMG
    for feature_idx in layer_top_features_img[layer]['indices']:
        if feature_idx < len(layer_concepts_img[layer]):
            predicted_concepts.add(layer_concepts_img[layer][feature_idx])

# Break down concepts into words for comparison
def concept_words(concept_list):
    words = set()
    for concept in concept_list:
        for word in concept.split():
            cleaned = re.sub(r'[^\w]', '', word.lower())
            if cleaned and len(cleaned) > 1:
                words.add(cleaned)
    return words

gt_words = concept_words(gt_concepts) if gt_concepts else set()
pred_words = concept_words(predicted_concepts)

# Print results
print(f"Image filename: {current_image_filename}")
print(f"Ground truth concepts: {gt_concepts}")
print(f"Predicted concepts: {sorted(predicted_concepts)}")

covered = gt_words.intersection(pred_words)
missed = gt_words - pred_words

print(f"\nCovered ground truth words ({len(covered)}): {sorted(covered)}")
print(f"Missed ground truth words ({len(missed)}): {sorted(missed)}")

## Evaluate on Entire Dataset

In [ ]:
# Load concept files for CLS tokens (existing)
concept_files_path_cls = "/dissect/CLS/BatchTopK-orig_top-128/ef-8/top1_concepts_gpt5-all"
layer_concepts_cls = {}

print("Loading CLS concept files...")
for layer in range(12):
    concept_file = f"{concept_files_path_cls}/layer_{layer}_top1_concepts.txt"
    try:
        with open(concept_file, 'r') as f:
            concepts = [line.strip() for line in f.readlines()]
        layer_concepts_cls[layer] = concepts
        print(f"CLS Layer {layer}: Loaded {len(concepts)} concepts")
    except FileNotFoundError:
        print(f"Warning: CLS concept file not found for layer {layer}")
        layer_concepts_cls[layer] = []

# Load concept files for Image tokens (you'll need to update the path)
concept_files_path_img = "/dissect/Image/BatchTopK-orig_top-128/ef-8/top1_concepts_meanpooled_gpt5-all"
layer_concepts_img = {}

print("Loading Image concept files...")
for layer in range(12):
    concept_file = f"{concept_files_path_img}/layer_{layer}_top1_concepts.txt"
    try:
        with open(concept_file, 'r') as f:
            concepts = [line.strip() for line in f.readlines()]
        layer_concepts_img[layer] = concepts
        print(f"Image Layer {layer}: Loaded {len(concepts)} concepts")
    except FileNotFoundError:
        print(f"Warning: Image concept file not found for layer {layer}")
        layer_concepts_img[layer] = []

print("Done loading all concept files!")

In [ ]:
import json
import re
from tqdm import tqdm

## Old testbed 
# def extract_image_id(filename):
#     """Extract image ID from filename like '000000533206.jpg' -> '533206'"""
#     match = re.search(r'(\d+)', filename)
#     if match:
#         return str(int(match.group(1)))
#     return filename

# def load_ground_truth_concepts(file_path, image_filename):
#     """Load ground truth concepts for a specific image"""
#     image_id = extract_image_id(image_filename)
    
#     with open(file_path, 'r') as f:
#         for line in f:
#             data = json.loads(line.strip())
#             possible_ids = [
#                 str(data.get('image_id', '')),
#                 str(data.get('id', '')),
#                 extract_image_id(data.get('image_path', '')),
#                 extract_image_id(data.get('filename', ''))
#             ]
            
#             if image_id in possible_ids:
#                 return data.get('concepts', [])
#     return None
## Old testbed 


## GPT-5 testbed
def extract_image_id(filename):
    """Extract image ID from filename like 'img_000005.jpeg' or full path."""
    # If full path, get basename
    base = os.path.basename(filename)
    # Remove extension
    name, _ = os.path.splitext(base)
    return name  # e.g., 'img_000005'

def load_ground_truth_concepts(file_path, image_filename):
    """Load ground truth concepts for a specific image (new dataset format)"""
    image_id = extract_image_id(image_filename)
    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line.strip())
            # Try to match by image_id (without extension)
            gt_image_id = extract_image_id(data.get('image_id', ''))
            if image_id == gt_image_id:
                return data.get('concepts', [])
    return None
## GPT-5 testbed


def extract_word_concepts_from_features_cls(layer_top_features, layer_concepts):
    """Extract and clean word concepts from CLS token features"""
    all_predicted_concepts = set()
    
    for layer in range(12):
        if layer in layer_top_features and layer in layer_concepts:
            top_features = layer_top_features[layer]
            concepts = layer_concepts[layer]
            
            for feature_idx in top_features['indices']:
                if feature_idx < len(concepts):
                    concept = concepts[feature_idx]
                    words = concept.split()
                    for word in words:
                        cleaned_word = re.sub(r'[^\w]', '', word.lower())
                        if cleaned_word and len(cleaned_word) > 1:
                            all_predicted_concepts.add(cleaned_word)
    
    return all_predicted_concepts

def extract_word_concepts_from_features_img(layer_top_features, layer_concepts):
    """Extract and clean word concepts from Image token features"""
    all_predicted_concepts = set()
    
    for layer in range(12):
        if layer in layer_top_features and layer in layer_concepts:
            top_features = layer_top_features[layer]
            concepts = layer_concepts[layer]
            
            for feature_idx in top_features['indices']:
                if feature_idx < len(concepts):
                    concept = concepts[feature_idx]
                    words = concept.split()
                    for word in words:
                        cleaned_word = re.sub(r'[^\w]', '', word.lower())
                        if cleaned_word and len(cleaned_word) > 1:
                            all_predicted_concepts.add(cleaned_word)
    
    return all_predicted_concepts

def calculate_coverage_for_image_union(image_path, gt_concepts_file):
    """Calculate coverage for a single image using union of CLS and Image token concepts"""
    try:
        # Load and process image
        image = Image.open(image_path).convert('RGB')
        embeddings = hooked_model.get_activations_from_image(image)

        # # Normalize embeddings to match training setup
        # normalized_embeddings = {}
        # for layer_name, act in embeddings.items():
        #     normalized_embeddings[layer_name] = act / act.norm(dim=-1, keepdim=True)
        # embeddings = normalized_embeddings
        
        # Extract top features for CLS tokens
        layer_top_features_cls = {}
        top_k = 20
        
        for layer in range(12):
            layer_name = f"layer-{layer}_resid-post"
            layer_activations = embeddings[layer_name]
            cls_activation = layer_activations[0, 0, :]  # CLS token
            
            sae_cls = saes_cls[layer]
            with torch.no_grad():
                feature_acts = sae_cls.encode(cls_activation.unsqueeze(0))
                feature_acts = feature_acts.squeeze(0)
                feature_acts[freq_idx_cls[layer]] = 0
                top_values, top_indices = torch.topk(feature_acts, top_k)
                
                layer_top_features_cls[layer] = {
                    'indices': top_indices.cpu().numpy(),
                    'values': top_values.cpu().numpy(),
                    'feature_acts': feature_acts.cpu().numpy()
                }
        
        # Extract top features for Image tokens
        layer_top_features_img = {}
        
        for layer in range(12):
            layer_name = f"layer-{layer}_resid-post"
            layer_activations = embeddings[layer_name]
            # Get all image tokens (tokens 1 to end, excluding CLS)
            img_activations = layer_activations[0, 1:, :]  # Shape: [seq_len-1, hidden_dim]
            
            sae_img = saes_img[layer]
            with torch.no_grad():
                # Encode each image token individually
                feature_acts_all_img = sae_img.encode(img_activations)  # [num_img_tokens, dict_size]
                # Average the feature activations
                feature_acts = feature_acts_all_img.mean(dim=0)  # [dict_size]
                feature_acts[freq_idx_img[layer]] = 0
                top_values, top_indices = torch.topk(feature_acts, top_k)
                
                layer_top_features_img[layer] = {
                    'indices': top_indices.cpu().numpy(),
                    'values': top_values.cpu().numpy(),
                    'feature_acts': feature_acts.cpu().numpy()
                }
        
        # Get predicted concepts from both CLS and Image tokens
        predicted_words_cls = extract_word_concepts_from_features_cls(layer_top_features_cls, layer_concepts_cls)
        predicted_words_img = extract_word_concepts_from_features_img(layer_top_features_img, layer_concepts_img)
        
        # Union the two sets
        predicted_words_union = predicted_words_cls.union(predicted_words_img)
        
        # Get ground truth concepts
        image_filename = os.path.basename(image_path)
        gt_concepts = load_ground_truth_concepts(gt_concepts_file, image_filename)
        
        if gt_concepts is None:
            return None, None, None, None, None, None
        
        # Break down ground truth into words
        gt_words = set()
        for concept in gt_concepts:
            words = concept.split()
            for word in words:
                cleaned_word = re.sub(r'[^\w]', '', word.lower())
                if cleaned_word and len(cleaned_word) > 1:
                    gt_words.add(cleaned_word)
        
        if len(gt_words) == 0:
            return None, None, None, None, None, None
        
        # Calculate metrics for union
        correctly_predicted = gt_words.intersection(predicted_words_union)
        coverage = len(correctly_predicted) / len(gt_words)
        precision = len(correctly_predicted) / len(predicted_words_union) if len(predicted_words_union) > 0 else 0
        
        return (coverage, precision, len(gt_words), len(predicted_words_union), 
                len(predicted_words_cls), len(predicted_words_img))
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None, None, None, None, None, None

In [ ]:
# Main evaluation loop for union approach
gt_concepts_file = "/concept_set/extracted_concepts_gpt5.jsonl"

results_union = []

print("Starting evaluation using UNION of CLS and Image token concepts...")
print("=" * 70)

for i, image_file in enumerate(tqdm(image_files, desc="Processing images")):
    image_path = os.path.join(image_folder, image_file)
    
    coverage, precision, gt_count, pred_count_union, pred_count_cls, pred_count_img = calculate_coverage_for_image_union(
        image_path, gt_concepts_file
    )
    
    if coverage is not None:
        results_union.append({
            'image': image_file,
            'coverage': coverage,
            'precision': precision,
            'gt_words': gt_count,
            'pred_words_union': pred_count_union,
            'pred_words_cls': pred_count_cls,
            'pred_words_img': pred_count_img
        })
        
        if (i + 1) % 1000 == 0:
            avg_coverage = sum(r['coverage'] for r in results_union) / len(results_union)
            avg_precision = sum(r['precision'] for r in results_union) / len(results_union)
            print(f"Processed {len(results_union)} images. Avg Coverage: {avg_coverage:.3f}, Avg Precision: {avg_precision:.3f}")

print(f"\nCompleted evaluation on {len(results_union)} images using union approach")

In [ ]:
# Calculate and display final statistics for union approach
if results_union:
    coverages = [r['coverage'] for r in results_union]
    precisions = [r['precision'] for r in results_union]
    gt_counts = [r['gt_words'] for r in results_union]
    pred_counts_union = [r['pred_words_union'] for r in results_union]
    pred_counts_cls = [r['pred_words_cls'] for r in results_union]
    pred_counts_img = [r['pred_words_img'] for r in results_union]
    
    print("\nFinal Results (UNION of CLS and Image token concepts):")
    print("=" * 60)
    print(f"Total images processed: {len(results_union)}")
    print(f"Images with ground truth: {len(results_union)}")
    print()
    print(f"Average Coverage (Recall): {np.mean(coverages):.3f} ± {np.std(coverages):.3f}")
    print(f"Average Precision: {np.mean(precisions):.3f} ± {np.std(precisions):.3f}")
    print(f"Median Coverage: {np.median(coverages):.3f}")
    print(f"Median Precision: {np.median(precisions):.3f}")
    print()
    print(f"Average GT words per image: {np.mean(gt_counts):.1f}")
    print(f"Average predicted words per image (Union): {np.mean(pred_counts_union):.1f}")
    print(f"Average predicted words per image (CLS only): {np.mean(pred_counts_cls):.1f}")
    print(f"Average predicted words per image (Image only): {np.mean(pred_counts_img):.1f}")
    
    # Compare with CLS-only results if available
    if 'results' in globals() and results:
        print(f"\nComparison with CLS-only results:")
        cls_coverages = [r['coverage'] for r in results]
        cls_precisions = [r['precision'] for r in results]
        
        print(f"CLS-only Average Coverage: {np.mean(cls_coverages):.3f}")
        print(f"Union Average Coverage: {np.mean(coverages):.3f}")
        print(f"Improvement: {np.mean(coverages) - np.mean(cls_coverages):.3f}")
        
        print(f"CLS-only Average Precision: {np.mean(cls_precisions):.3f}")
        print(f"Union Average Precision: {np.mean(precisions):.3f}")
        print(f"Precision change: {np.mean(precisions) - np.mean(cls_precisions):.3f}")

else:
    print("No results to analyze - no images had matching ground truth concepts")

In [ ]:
# Get top N images by coverage
top_n = 20  # Change this to get more or fewer images

# Sort results_union by coverage (descending)
sorted_results = sorted(results_union, key=lambda x: x['coverage'], reverse=True)

print(f"Top {top_n} images by coverage:")
for i, result in enumerate(sorted_results[:top_n], 1):
    print(f"{i:2d}. Image: {result['image']} | Coverage: {result['coverage']:.3f} | Precision: {result['precision']:.3f}")